# Data Exploration

In [ ]:
# Suppress Possible Warnings : In some installations a warning due to pyg_lib path can arise: not relevant to our setting
import warnings
warnings.filterwarnings(
    "ignore",
    message=".*An issue occurred while importing 'pyg-lib'.*",
    category=UserWarning,
    module="torch_geometric.typing"
)
warnings.filterwarnings(
    "ignore",
    message=".*An issue occurred while importing 'torch-sparse'.*",
    category=UserWarning,
    module="torch_geometric.typing"
)

## Import packages

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib as mpl
import seaborn as sns
import matplotlib.pyplot as plt

# Load SWaT dataset
Secure Water Treatment (SWaT) is a fully operational water treatment plant located in Singapore. The orignial data resolution is 1 second, but it was downsampled to 10 seconds for this workshop following the work of GDN.

It contains:
- 11 days continuous operations
- 7 days under normal operations
- 4 days were under attack scenarios
- 41 attacks in total


In [ ]:
# print os gwd
import os
print(os.getcwd())

In [ ]:
df = pd.read_csv(f'run/datasets/swat/test.csv', index_col=0)
df.index = pd.to_datetime(df.index)

In [ ]:
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
# Dataset Start Time and End Time
df_time_start = df.index[0]
df_time_end = df.index[-1]

# Visulization Strat Time and End Time
time_start =  pd.to_datetime('2015-12-29T00:00:00')
time_end =pd.to_datetime('2015-12-30T00:00:00')
time_len = int((time_end - time_start) / np.timedelta64(1, 's'))

def get_index(time):
    time = pd.to_datetime(time)
    return int((time - df_time_start) / np.timedelta64(10, 's'))

idx_start = get_index(time_start)
idx_end = get_index(time_end)
print(idx_start, idx_end)

# Visulization

Since the whole dataset time line is too long, here we only visulize 29 Dec 2015's data, you can easily change to what date you want to visulize by modifing start and end indexes.

## Set Anomaly Time Regions

We can get the information about attacks from SWaT dataset `SWaT.A1 & A2_Dec 2015/List_of_attacks_Final.pdf`. Here are attacks happened on 29 Dec 2015.

- `anomaly_feature` means this attack happened on what part of the system.
- `anomaly_time_start` and `anomaly_time_end` mark the duration of that attack.

Since the anomaly recording file is in bad format, so here I had to write down all attacks manually. You can still create a script to do this.

In [ ]:
anomaly_feature = ['MV304', 'MV303', 'LIT301', 'MV303', 'AIT504', 'AIT504', 'MV101', 'LIT101', 'UV401', 'AIT502', 'P501']
anomaly_time_start = [np.array('2015-12-29T11:11:25', dtype=np.datetime64),
                      np.array('2015-12-29T11:35:40', dtype=np.datetime64),
                      np.array('2015-12-29T11:57:25', dtype=np.datetime64),
                      np.array('2015-12-29T14:38:12', dtype=np.datetime64),
                      np.array('2015-12-29T18:15:01', dtype=np.datetime64),
                      np.array('2015-12-29T18:15:43', dtype=np.datetime64),
                      np.array('2015-12-29T18:30:00', dtype=np.datetime64),
                      np.array('2015-12-29T18:30:00', dtype=np.datetime64),
                      np.array('2015-12-29T22:55:18', dtype=np.datetime64),
                      np.array('2015-12-29T22:55:18', dtype=np.datetime64),
                      np.array('2015-12-29T22:55:18', dtype=np.datetime64)
                      ]
anomaly_time_end = [np.array('2015-12-29T11:15:17', dtype=np.datetime64),
                    np.array('2015-12-29T11:42:50', dtype=np.datetime64),
                    np.array('2015-12-29T12:02:00', dtype=np.datetime64),
                    np.array('2015-12-29T14:50:08', dtype=np.datetime64),
                    np.array('2015-12-29T18:15:01', dtype=np.datetime64),
                    np.array('2015-12-29T18:22:17', dtype=np.datetime64),
                    np.array('2015-12-29T18:42:00', dtype=np.datetime64),
                    np.array('2015-12-29T18:42:00', dtype=np.datetime64),
                    np.array('2015-12-29T23:03:00', dtype=np.datetime64),
                    np.array('2015-12-29T23:03:00', dtype=np.datetime64),
                    np.array('2015-12-29T23:03:00', dtype=np.datetime64)
                    ]
assert len(anomaly_feature) == len(anomaly_time_start) and len(anomaly_feature) == len(anomaly_time_end)

In [ ]:
# needed for the exercise later
ad_idx = anomaly_feature.index('MV101')
get_index(anomaly_time_start[ad_idx])

In [ ]:
# get each alarm's duration
for i in range(len(anomaly_feature)):
    start = (anomaly_time_start[i])
    end = (anomaly_time_end[i])
    print(f'{anomaly_feature[i]}: {start} - {end} = {end - start}')

# SWaT Testbed Process Overview
![SWaT Process](https://drive.google.com/uc?export=view&id=1k8OqD47xfakwf5Ap90RQBYW4giml34uP)

# Plot Signals with Anomaly Regions

### Visualizing only signals with anomaly regions

In [ ]:
def plot_attack_regions(selected_features, time_start=time_start, time_end=time_end):
    df_selected = df.loc[time_start:time_end]

    axes = df_selected.plot(y=selected_features, subplots=True, figsize=(12, 0.6*len(selected_features)), color='#2f83e4')

    for ad_feat, ad_start, ad_end in zip(anomaly_feature, anomaly_time_start, anomaly_time_end):
        mask = (df_selected.index >= ad_start) & (df_selected.index <= ad_end)
        if len(df_selected.loc[mask]) == 0 or ad_feat not in selected_features:
            continue
        df_selected.loc[mask, ad_feat].plot(ax=axes[selected_features.index(ad_feat)], color='tomato', linewidth=2)

    plt.show()

In [ ]:
# TODO: plot anomaly_feature and attack
plot_attack_regions(selected_features=...)

In [ ]:
# plot all features
plot_attack_regions(selected_features=df.columns.tolist())

In [ ]:
# TODO: Inspect features of your interest
plot_attack_regions(selected_features=...)

### Can you spot which signals are under attack?

In [ ]:
# Visulization Strat Time and End Time
time_start =  pd.to_datetime('2015-12-28T00:00:00')
time_end =pd.to_datetime('2015-12-29T00:00:00')
plot_attack_regions(selected_features=list(set(anomaly_feature))+['attack'], time_start=time_start, time_end=time_end)

# Analysis

Play around with the data and see if you can find any interesting patterns or insights.

In [ ]:
# TODO: draw correlation matrix
corr = ...
plt.figure(figsize=(10,10))
sns.heatmap(corr, vmax=1, vmin=-1, square=True, annot=False, cmap='coolwarm')
plt.title('Correlation between features')
plt.show()
